# Human-Agent Collaboration

**Level:** Advanced · **Time:** 60 min

In previous modules, we discussed Agent Orchestration and Durable Execution. Here, we apply those concepts to **Human-in-the-Loop (HITL)** workflows.

HITL is not just a button; it is an interaction contract. In this notebook, we will demonstrate two specific paradigms using **LangGraph**:
1. **Static Breakpoints (`interrupt_before`)**: Pausing a workflow before a high-risk action.
2. **Dynamic Breakpoints (`interrupt()`)**: Letting an agent autonomously pause to ask for human clarification or co-authoring.

> **Note:** The code blocks are written in syntactically correct LangGraph. To allow this notebook to run without local API keys or complex database checkpointers, the outputs are simulated.

---
## Pattern 1: Static Breakpoints (Approval)

![Risk Framework](../../../assets/hitl_risk_matrix.svg)

For **High** and **Critical** risk actions (like executing a database rollback or disabling a feature flag), the system must explicitly pause. 

In LangGraph, this is achieved using a **State Checkpointer** (to save the memory) and `interrupt_before` (to pause the graph before it enters the action node).

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class AgentState(TypedDict):
    incident: str
    proposed_action: str
    action_result: str

def analyze_node(state: AgentState):
    print("[Agent] Analyzing incident...")
    return {"proposed_action": "Rollback deploy-842"}

def execute_action_node(state: AgentState):
    print(f"[System] Executing High-Risk Action: {state['proposed_action']}")
    return {"action_result": "Success"}

# 1. Setup the Graph
workflow = StateGraph(AgentState)
workflow.add_node("Analyze", analyze_node)
workflow.add_node("ExecuteAction", execute_action_node)

workflow.add_edge(START, "Analyze")
workflow.add_edge("Analyze", "ExecuteAction")
workflow.add_edge("ExecuteAction", END)

# 2. Setup the Checkpointer (Persistence)
memory = MemorySaver()

# 3. Compile with a STATIC Breakpoint
app = workflow.compile(
    checkpointer=memory, 
    interrupt_before=["ExecuteAction"] # Pauses BEFORE this node
)

# --- Execution ---
thread = {"configurable": {"thread_id": "incident-104"}}

# Run 1: Agent analyzes, then pauses before ExecuteAction
# result = app.invoke({"incident": "EU Checkout Drop"}, config=thread)

# Run 2: Human approves, graph resumes
# result_final = app.invoke(None, config=thread)

--- RUN 1 (Initial Trigger) ---
[Agent] Analyzing incident...
[System] Graph Execution Suspended.
State saved in Checkpointer. Next node to execute: 'ExecuteAction'

--- HUMAN REVIEW ---
Human reviews proposal: "Rollback deploy-842"
Human clicks "Approve"

--- RUN 2 (Resuming Graph) ---
[System] Resuming from Checkpoint...
[System] Executing High-Risk Action: Rollback deploy-842


**Selection Criteria:** Use `interrupt_before` for mandatory compliance, security, or safety checks where a specific tool/action *must never* run autonomously.

---
## Pattern 2: Dynamic Escalation & Co-authoring (`interrupt()`)

![HITL Workflow](../../../assets/hitl_workflow.svg)

Sometimes, the agent doesn't know it needs to pause until it is mid-execution. Perhaps it lacks confidence, or perhaps the workflow design is "Agent drafts an email, Human edits it, Agent sends it."

LangGraph's dynamic `interrupt()` function allows a node to pause itself, send a payload to the user, and wait for a `Command(resume=...)` response.

In [ ]:
from langgraph.types import interrupt, Command

def draft_communication_node(state: AgentState):
    print("[Agent] Drafting customer communication...")
    draft = "Subject: EU Checkout Outage\nWe experienced a brief outage. It is now resolved."
    
    # 1. Dynamic Interrupt: Pause and send draft to human
    print("[System] Interrupting for human feedback...")
    human_feedback = interrupt({"draft": draft})
    
    # 2. Agent receives the human's edits and continues
    print(f"[Agent] Received Human Feedback: {human_feedback}")
    
    final_draft = draft + "\nUpdate: " + human_feedback
    return {"proposed_action": final_draft}

# --- Execution ---
# Run 1: Agent pauses dynamically
# result = app.invoke({"incident": "EU Checkout Drop"}, config=thread)

# Run 2: Human resumes with feedback via Command
# result_final = app.invoke(
#     Command(resume="Add a 20% discount code: SORRY20"), 
#     config=thread
# )

--- RUN 1 ---
[Agent] Drafting customer communication...
[System] Interrupting for human feedback...
Graph Suspended. Payload sent to Human: {'draft': 'Subject: EU Checkout Outage...'}

--- HUMAN REVIEW ---
Human edits the draft and submits feedback.

--- RUN 2 (Resume with Command) ---
[System] Resuming...
[Agent] Received Human Feedback: Add a 20% discount code: SORRY20
[Agent] Finalizing Draft.


---
## HITL Anti-Patterns

While implementing Human-in-the-Loop, beware of these three critical failure modes:

### 1. State Leakage (Amnesia)
If you do not persist state before pausing, the agent will forget everything when the human finally responds hours later.

In [ ]:
from langgraph.graph import StateGraph

# ANTI-PATTERN: No Checkpointer
# When compiling with an interrupt, LangGraph strictly requires a checkpointer.
# If omitted, it raises a ValueError to prevent State Leakage in production.
workflow_bad = StateGraph(AgentState)
workflow_bad.add_node("Analyze", analyze_node)
workflow_bad.add_node("ExecuteAction", execute_action_node)
workflow_bad.add_edge(START, "Analyze")
workflow_bad.add_edge("Analyze", "ExecuteAction")
workflow_bad.add_edge("ExecuteAction", END)

try:
    # Missing checkpointer=memory!
    app_bad = workflow_bad.compile(interrupt_before=["ExecuteAction"]) 
except ValueError as e:
    print(f"CRASH AVERTED: {e}")

CRASH AVERTED: Compile with 'interrupt_before' requires a checkpointer\n

### 2. Rubber Stamping (Opaque Handoffs)
If you don't provide the *why*, the human will just click approve to get the alert out of their queue.

In [ ]:
# ANTI-PATTERN: Opaque Handoff
def agent_node_bad(state):
    # The human just sees "Action: Rollback". They have no idea what caused it.
    return {"proposed_action": "Rollback deploy-842"} 

# BETTER: Explainable Handoff
def agent_node_better(state):
    return {
        "proposed_action": "Rollback deploy-842",
        "reasoning": "Deploy-842 correlates 100% with a 31% drop in EU checkout.",
        "evidence_links": ["https://datadog.internal/dashboard/123"]
    }

print("Bad Handoff Payload:", agent_node_bad({}))
print("Good Handoff Payload:", agent_node_better({}))

Bad Handoff Payload: {'proposed_action': 'Rollback deploy-842'}\nGood Handoff Payload: {'proposed_action': 'Rollback deploy-842', 'reasoning': 'Deploy-842 correlates 100% with a 31% drop in EU checkout.', 'evidence_links': ['https://datadog.internal/dashboard/123']}\n

### 3. Polling vs Event-Driven Wakeups
Agents should not sit in a `while True` loop consuming CPU, and humans should not have to manually refresh a dashboard to see if an agent is stuck.

In [ ]:
import time

# ANTI-PATTERN: CPU Polling
class MockDB:
    def __init__(self):
        self.checks = 0
    def check_human_approved(self):
        self.checks += 1
        return self.checks > 2

db = MockDB()

def wait_for_human():
    print("Agent is polling...")
    while not db.check_human_approved():
        print("Still waiting... Wasting thread resources!")
        time.sleep(1) 
    print("Human finally approved.")

wait_for_human()

# BETTER: Event-Driven (LangGraph)
# The graph yields execution entirely. The human clicking a button 
# in the UI triggers a webhook that calls `app.invoke(None, config=thread)` to wake it up.

Agent is polling...\nStill waiting... Wasting thread resources!\nStill waiting... Wasting thread resources!\nHuman finally approved.\n

---
## The Handoff Packet & Explainability

When designing your HITL interface (the UI the human actually sees when the graph is paused, like a Slack message or a web dashboard), ensure the **Handoff Packet** is explainable. A human cannot safely approve a high-risk action if they don't know *why* the agent proposed it.

Always include:
1. **The Proposed Action**: Exactly what the agent intends to do.
2. **The Reason**: Why the agent thinks this is the correct action.
3. **Links to Evidence (Provenance)**: The dashboard, logs, or metrics that led to the conclusion.
4. **Confidence/Alternatives**: What else was considered?

Below are examples of comprehensive Handoff Packets for different use-cases.

In [ ]:
import json

# Use Case 1: Incident Rollback (Critical Risk)
incident_handoff = {
    "workflow": "Northstar Incident Mitigation",
    "proposed_action": "Execute Rollback of deploy-842 in eu-west-1",
    "reasoning": "Deploy-842 correlates exactly with a 31% drop in successful checkouts. 504 Gateway errors spiked from 0.01% to 15% immediately after the rollout.",
    "evidence": [
        "https://datadog.internal/dashboard/checkout-eu",
        "https://github.com/northstar/api/deploy/842"
    ],
    "confidence": 0.99,
    "human_options": ["Approve Rollback", "Reject", "Modify Rollback Target"]
}

# Use Case 2: Draft Customer Communication (High Risk)
email_handoff = {
    "workflow": "Customer Success Outreach",
    "proposed_action": "Send 4,500 apology emails via SendGrid",
    "reasoning": "These 4,500 users experienced checkout failures during the 30-minute outage. Policy dictates an apology and a 20% discount code.",
    "draft_preview": "Subject: EU Checkout Outage... [Click to expand]",
    "confidence": 0.85,
    "human_options": ["Approve and Send", "Edit Draft", "Cancel"]
}

print("=== Incident Rollback Handoff Packet ===")
print(json.dumps(incident_handoff, indent=2))
print("\n=== Customer Communication Handoff Packet ===")
print(json.dumps(email_handoff, indent=2))

=== Incident Rollback Handoff Packet ===
{
  "workflow": "Northstar Incident Mitigation",
  "proposed_action": "Execute Rollback of deploy-842 in eu-west-1",
  "reasoning": "Deploy-842 correlates exactly with a 31% drop in successful checkouts. 504 Gateway errors spiked from 0.01% to 15% immediately after the rollout.",
  "evidence": [
    "https://datadog.internal/dashboard/checkout-eu",
    "https://github.com/northstar/api/deploy/842"
  ],
  "confidence": 0.99,
  "human_options": [
    "Approve Rollback",
    "Reject",
    "Modify Rollback Target"
  ]
}

=== Customer Communication Handoff Packet ===
{
  "workflow": "Customer Success Outreach",
  "proposed_action": "Send 4,500 apology emails via SendGrid",
  "reasoning": "These 4,500 users experienced checkout failures during the 30-minute outage. Policy dictates an apology and a 20% discount code.",
  "draft_preview": "Subject: EU Checkout Outage... [Click to expand]",
  "confidence": 0.85,
  "human_options": [
    "Approve and Send